# Phase 3: Leakage-Safe Classification Baseline

This notebook evaluates three supervised-learning algorithms on the Phase 2 household dataset: Logistic Regression, Random Forest, and Gradient Boosting. Every model uses the same stratified train/test split and the same preprocessing pipeline. Missing-value imputation, categorical encoding, and numerical scaling are fitted on the training data inside each model pipeline.


## Evaluation design

- **Unit of analysis:** one agricultural household.
- **Target:** `1` for agricultural-input use and `0` for non-use.
- **Predictors:** household context, land and crops, agricultural practices, and aggregated livestock variables.
- **Excluded:** `idquest` and all direct `S5` agricultural-input response fields, which could leak the target.
- **Split:** 75% training and 25% testing, stratified by the target, with `random_state=42`.
- **Metrics:** accuracy, precision, recall, F1-score, and ROC-AUC.
- **Baseline definition:** models use default class weighting, so later class balancing or tuning can be evaluated as a separate improvement.

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path('prepared_household_data.csv')
RESULTS_PATH = Path('phase_3_baseline_results.csv')

raw = pd.read_csv(DATA_PATH, low_memory=False)
leak_columns = [column for column in raw.columns if column.startswith('s5q')]
exclude_columns = {'idquest', 'target'}
feature_columns = [
    column for column in raw.columns
    if column not in exclude_columns and column not in leak_columns
 ]

X = raw[feature_columns].copy()
y = raw['target'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_columns = X.select_dtypes(include=['number']).columns.tolist()
categorical_columns = [column for column in X.columns if column not in numeric_columns]

print(f'Rows: {len(raw):,}')
print(f'Predictors used: {len(feature_columns)}')
print(f'Direct S5 fields excluded: {len(leak_columns)}')
print(f'Train/test rows: {len(X_train):,}/{len(X_test):,}')
print(f'Target distribution: {y.value_counts().sort_index().to_dict()}')

In [ ]:
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                'numeric',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]),
                numeric_columns,
            ),
            (
                'categorical',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ]),
                categorical_columns,
            ),
        ],
        remainder='drop',
    )

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, random_state=42, min_samples_leaf=2, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

def evaluate_model(name, model):
    pipeline = Pipeline([
        ('preprocessor', build_preprocessor()),
        ('model', model),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    return pipeline, {
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, probabilities),
    }

## Model comparison and baseline output

Each model is fitted independently with the same train-only preprocessing logic. The primary comparison metric is F1-score because the target is moderately imbalanced and both types of classification error matter. ROC-AUC is also reported because it evaluates ranking quality across probability thresholds.

In [ ]:
fitted_pipelines = {}
results = []

for model_name, model in models.items():
    pipeline, metrics = evaluate_model(model_name, model)
    fitted_pipelines[model_name] = pipeline
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
results_df.to_csv(RESULTS_PATH, index=False)
display(results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'ROC_AUC': '{:.4f}',
}))
print(f'Saved baseline results to {RESULTS_PATH}')
print(f'Best baseline model by F1: {results_df.loc[0, "Model"]}')

## Phase 3 conclusion and handoff

The leakage-safe baseline comparison is complete. All three models used the same 75/25 stratified split and fitted imputation and encoding within their pipelines.


| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Gradient Boosting | 0.7980 | 0.8117 | 0.9210 | **0.8629** | 0.8546 |
| Random Forest | 0.7945 | 0.8050 | 0.9267 | 0.8616 | 0.8536 |
| Logistic Regression | 0.7968 | 0.8227 | 0.8993 | 0.8593 | 0.8448 |

Gradient Boosting is the baseline leader by F1-score and will be the primary candidate for Phase 4 improvement. The differences are small, so the final discussion should avoid claiming a decisive winner without considering the project objective and the trade-off between recall and precision.


**Handoff to Phase 4:** evaluate one documented improvement method, such as class weighting, SMOTE, or hyperparameter tuning, against this fixed baseline using the same test set and metrics.